In [1]:
# Numerical and data handling
import numpy as np
import pandas as pd

# Pytorch
import torch

# PDF reading
import pymupdf

# FAISS for vector search
import faiss

# Core Python libraries
import unicodedata
import re
from typing import Any

In [2]:
path = r"C:\Users\user\Downloads\Bert.pdf"

In [3]:
doc = pymupdf.open(path)

In [4]:
print(type(doc))

<class 'pymupdf.Document'>


In [5]:
def extract_text(pdf_path : str) -> list[dict[str, Any]]:
    """
    Extracts the text from the document page-wise

    Returns:
        A List of Dictionaries:
        Each Dictionary contains
        - page_number
        - text
        - character_count
        - word_count
    """

    document = pymupdf.open(pdf_path)

    pages = []

    for page_idx in range(len(document)):
        text = document[page_idx].get_text()

        page_data = {
            "page_number" : page_idx + 1,
            "text" : text,
            "character_number" : len(text),
            "word_count" : len(text.split())
        }

        pages.append(page_data)

    return pages

In [6]:
def clean_data(text: str) -> str:
    """"
    Improvised PDF text cleaning.

    Handles:
    - Unicode normalization
    - Common PDF ligatures
    - Hyphenated line breaks
    - Extra newlines
    - Extra spaces
    - Broken spacing around punctuation
    """

    if text is None:
        return ""

    # Unicode normalisation
    text = unicodedata.normalize("NFKC", text)

    # Fix common PDF ligatures
    ligature_map = {
        "ﬁ": "fi",
        "ﬂ": "fl",
        "ﬀ": "ff",
        "ﬃ": "ffi",
        "ﬄ": "ffl"
    }

    for bad, good in ligature_map.items():
        text = text.replace(bad,good)

    # Fix hyphenated words broken across lines:
    # Example: "trans-\nformer" -> "transformer"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)

    # Replace remaining newlines, carriage returns, and tabs with spaces
    text = re.sub(r"[\n\r\t]+", " ", text)

    # Remove repeated spaces
    text = re.sub(r"\s+", " ", text)

    # Fix spaces before punctuation
    text = re.sub(r"\s+([.,;:!?%)\]])", r"\1", text)

    # Fix spaces after opening brackets
    text = re.sub(r"([(\[])\s+", r"\1", text)

    text = text.strip()

    return text

In [7]:
pages = extract_text(path)

cleaned_page_data = []

for page in pages:
    cleaned_text = clean_data(page["text"])

    cleaned_page_data.append({
        "page_number" : page["page_number"],
        "text" : cleaned_text,
        "character_count" : len(cleaned_text),
        "word_count" : len(cleaned_text.split())
    })

In [8]:
print(f"Number of cleaned pages: {len(cleaned_page_data)}")

Number of cleaned pages: 16


In [12]:
for i in cleaned_page_data:
    print()
    print(i["page_number"])
    print(i["character_count"])


1
3911

2
4477

3
3655

4
4805

5
3810

6
4428

7
3680

8
4451

9
4080

10
4492

11
4716

12
3859

13
3660

14
4048

15
2320

16
3110


In [15]:
def chunker(pages: list[dict[str, Any]], chunk_size: int = 350, overlap: int = 40):
    """
    Creates overlapping word-based chunks from page-wise PDF text.

    Args:
        pages:
            A list of dictionaries containing page_number and text.
        
        chunk_size:
            The number of words in each chunk.

        overlap:
            The number of words that overlap between the chunks.

    Returns:
        A list of chunk dictionaries

    """
    if overlap > chunk_size:
        raise ValueError("Chunk size must be greater than overlap.")

    chunks = []
    chunk_id = 0

    
    for page in pages:
        text = page["text"]
        page_number = page["page_number"]
        words = text.split()

        if len(text) == 0:
            continue

        start = 0

        while start < len(words):
            end = start + chunk_size
            chunk_words = text[start : end]
            chunk = "".join(chunk_words)

            if len(chunk_words) > 0:
                chunks.append({
                    "chunk_id": chunk_id,
                    "page_number": page_number,
                    "words" : chunk,
                    "word_count": len(chunk_words),
                    "character_count": len(chunk),
                    "preview": chunk[:300]
                })

                chunk_id += 1
            start = end - overlap
    return chunks

In [16]:
chunks_data = chunker(
    pages = cleaned_page_data,
    overlap = 40,
    chunk_size = 300
)

In [3]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

In [4]:
model_name = "distilbert-base-uncased-distilled-squad"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

print("Model loaded successfully", model_name)

e:\Projects\QnA_Model\QnA_Model\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--distilbert-base-uncased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 102/102 [00:00<00:00, 14356.34it/s]

Model loaded successfully distilbert-base-uncased-distilled-squad


In [ ]:
def clean_answer_display(answer: str) -> str:
    """
    Cleans the final output for display

    Handles:
    - Special tokens
    - Wordpiece artifacts
    - Extra spaces
    - Broken Punctuation Spacing
    """
